In [ ]:
# ==============================================================================
# ML PRACTICAL 5: K-MEANS CLUSTERING
# Objective: Customer / Sales transaction segmentation using K-Means Clustering
# ==============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set clean visualization theme
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 2. Load Dataset
df = pd.read_csv('sales_data_sample.csv', encoding='unicode_escape')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows of selected columns:")
print(df[['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'SALES', 'ORDERDATE', 'DEALSIZE']].head())

# 3. Feature Selection & Standardization
# Retain numerical clustering columns (preserves 100% of rows without dropping international orders)
X = df[['SALES', 'QUANTITYORDERED', 'PRICEEACH']].copy()
print("\nMissing values in clustering features:\n", X.isnull().sum())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"\nScaled Feature Matrix Shape: {X_scaled.shape}")
print(pd.DataFrame(X_scaled, columns=['SALES_scaled', 'QUANTITY_scaled', 'PRICE_scaled']).head())

# 4. Elbow Method & Silhouette Analysis (Finding Optimal K)
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

sil_scores = []
sil_k_range = range(2, 11)

for k in sil_k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

# 2-Panel Diagnostic Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Elbow Plot
axes[0].plot(list(k_range), wcss, marker='o', color='#1f77b4', lw=2.5, markersize=7)
axes[0].axvline(x=4, color='#d62728', linestyle='--', label='Optimal K = 4 (Elbow)')
axes[0].scatter(4, wcss[3], color='#d62728', s=120, zorder=5)
axes[0].set_title('Elbow Method (Inertia vs K)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[0].set_ylabel('Within-Cluster Sum of Squares (Inertia)', fontsize=11)
axes[0].set_xticks(list(k_range))
axes[0].legend(frameon=True)

# 2. Silhouette Score Plot
axes[1].plot(list(sil_k_range), sil_scores, marker='s', color='#2ca02c', lw=2.5, markersize=7)
axes[1].axvline(x=4, color='#d62728', linestyle='--', label=f'Peak Score at K=4 ({sil_scores[2]:.4f})')
axes[1].scatter(4, sil_scores[2], color='#d62728', s=120, zorder=5)
axes[1].set_title('Silhouette Score vs Number of Clusters (K)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[1].set_ylabel('Mean Silhouette Score', fontsize=11)
axes[1].set_xticks(list(sil_k_range))
axes[1].legend(frameon=True)

plt.tight_layout()
plt.show()

# 5. Fit Final K-Means Model (K = 4)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
X['Cluster'] = kmeans.fit_predict(X_scaled)

# Inverse transform cluster centers to original scale for business interpretation
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
centroids_df = pd.DataFrame(centroids_orig, columns=['SALES', 'QUANTITYORDERED', 'PRICEEACH'])
centroids_df.index.name = 'Cluster'

print("\nCluster Centroids (Original Units):")
print(centroids_df.round(2))

# 6. Cluster Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

# 1. Sales vs Quantity Ordered
sns.scatterplot(
    data=X, x='QUANTITYORDERED', y='SALES', hue='Cluster',
    palette=palette, alpha=0.6, s=50, ax=axes[0]
)
axes[0].scatter(
    centroids_df['QUANTITYORDERED'], centroids_df['SALES'],
    color='black', s=220, marker='X', edgecolor='white', linewidth=1.5,
    label='Centroids', zorder=10
)
axes[0].set_title('Cluster Segmentation: Sales vs Quantity Ordered', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity Ordered', fontsize=11)
axes[0].set_ylabel('Sales ($)', fontsize=11)
axes[0].legend(title='Cluster', frameon=True)

# 2. Sales vs Unit Price
sns.scatterplot(
    data=X, x='PRICEEACH', y='SALES', hue='Cluster',
    palette=palette, alpha=0.6, s=50, ax=axes[1]
)
axes[1].scatter(
    centroids_df['PRICEEACH'], centroids_df['SALES'],
    color='black', s=220, marker='X', edgecolor='white', linewidth=1.5,
    label='Centroids', zorder=10
)
axes[1].set_title('Cluster Segmentation: Sales vs Price Each', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Price Each ($)', fontsize=11)
axes[1].set_ylabel('Sales ($)', fontsize=11)
axes[1].legend(title='Cluster', frameon=True)

plt.tight_layout()
plt.show()

# 7. Cluster Profiling & Business Segmentation
profile = X.groupby('Cluster').agg(
    Order_Count=('SALES', 'count'),
    Avg_Sales=('SALES', 'mean'),
    Avg_Quantity=('QUANTITYORDERED', 'mean'),
    Avg_Price=('PRICEEACH', 'mean')
).reset_index()

profile['Share_%'] = (profile['Order_Count'] / len(X) * 100).round(1)

# Map meaningful persona names
persona_map = {
    0: 'Bulk Economy Orders (High Volume, Low Price)',
    1: 'High-Value VIP Orders (High Volume, Premium Price)',
    2: 'Premium Standard Orders (Moderate Volume, High Price)',
    3: 'Budget Retail Orders (Low Volume, Low Price)'
}
profile['Segment_Persona'] = profile['Cluster'].map(persona_map)

print("\n--- Cluster Business Profiles ---")
print(profile.round(2))

# ==============================================================================
# VIVA QUICK REFERENCE:
# 1. Elbow Method: Plots WCSS (Inertia) vs K. Choose the point where rate of decrease slows.
# 2. Silhouette Score: Measures separation distance between clusters (ranges from -1 to 1).
# 3. Feature Scaling: Crucial because K-Means is Euclidean distance-based; unscaled large
#    features would dominate distance metrics.
# ==============================================================================